In [10]:
%pip install langchain-groq

  Using cached langchain_groq-1.1.3-py3-none-any.whl.metadata (2.9 kB)
  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
Using cached langchain_groq-1.1.3-py3-none-any.whl (20 kB)
Using cached groq-0.37.1-py3-none-any.whl (137 kB)

   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq


c:\Users\geetikak\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [13]:
# This defines the expected structure and types of the dictionary.
class State(TypedDict):
    topic: str
    draft: str
    critique: str
    attempts: Annotated[int, operator.add]
    # operator.add is the function version of '+' operator.

In [ ]:
def	write(state:	State)	->	dict:
    prompt = f"Write one punchy sentence about:	{state['topic']}" 
    if state.get("critique"): 
        prompt	+=	f"\nAddress	this critique:	{state['critique']}" 

    draft = llm.invoke(prompt).content 
    return	{"draft":	draft,	"attempts":	1}

In [15]:
def critique(state: State) -> dict:
    prompt	=	f"Critique	this	sentence	in	5	words	or	fewer,	or	say	'GOOD':\n{state['draft']}" 
    return	{"critique":	llm.invoke(prompt).content}

In [24]:
def	should_continue(state:	State)	->	str: 
    if	"GOOD"	in	state["critique"]	or	state["attempts"]	>=	3:
        return	END 
    return	"write"

In [29]:
builder = StateGraph(State)

builder.add_node("write", write)
builder.add_node("critique", critique)

builder.add_edge(START, "write")
builder.add_edge("write", "critique")
builder.add_conditional_edges("critique", should_continue, {
    "write": "write",
    END: END
})


In [30]:
graph = builder.compile()

In [33]:
final = graph.invoke({"topic": "OpanAI", "attempts": 2})
print(final['draft'])
print(f"took {final['attempts']} attempt(s)")

OpenAI is revolutionizing the tech landscape with its cutting-edge AI models, including ChatGPT, that are transforming the way humans interact, create, and innovate.
took 3 attempt(s)
